In [1]:
# ---- Core imports ----
import os
from pathlib import Path
from collections import Counter
import random

# ---- Image validation ----
from PIL import Image

# ---- Reproducibility ----
random.seed(42)

# ---- Paths (relative, safe) ----
# Path.cwd() = current working directory of the notebook runtime.
# notebook is inside /notebooks,go one level up to project root.
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_DIR / "IMG_CLASSES"     # your raw dataset folder
OUTPUT_DIR = PROJECT_DIR / "data"         # where train/val/test will be created

print("PROJECT_DIR:", PROJECT_DIR.name)   # prints only folder name, not full path
print("RAW_DIR exists?", RAW_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR.name)


PROJECT_DIR: model_training
RAW_DIR exists? True
OUTPUT_DIR: data


In [2]:
# List class folders (subdirectories inside IMG_CLASSES)
class_dirs = [p for p in RAW_DIR.iterdir() if p.is_dir()]

print(f"Found {len(class_dirs)} class folders.\n")

# Print the folder names (these become labels)
for p in class_dirs:
    print("-", p.name)


Found 10 class folders.

- 1. Eczema 1677
- 10. Warts Molluscum and other Viral Infections - 2103
- 2. Melanoma 15.75k
- 3. Atopic Dermatitis - 1.25k
- 4. Basal Cell Carcinoma (BCC) 3323
- 5. Melanocytic Nevi (NV) - 7970
- 6. Benign Keratosis-like Lesions (BKL) 2624
- 7. Psoriasis pictures Lichen Planus and related diseases - 2k
- 8. Seborrheic Keratoses and other Benign Tumors - 1.8k
- 9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k


In [3]:
# Count images per class folder
image_exts = {".jpg", ".jpeg", ".png", ".webp"}

counts = {}
for cls in class_dirs:
    imgs = [p for p in cls.rglob("*") if p.suffix.lower() in image_exts]
    counts[cls.name] = len(imgs)

# Show sorted counts (largest first)
for k, v in sorted(counts.items(), key=lambda x: x[1], reverse=True):
    print(f"{k:45} -> {v}")


5. Melanocytic Nevi (NV) - 7970               -> 7970
4. Basal Cell Carcinoma (BCC) 3323            -> 3323
2. Melanoma 15.75k                            -> 3140
10. Warts Molluscum and other Viral Infections - 2103 -> 2103
6. Benign Keratosis-like Lesions (BKL) 2624   -> 2079
7. Psoriasis pictures Lichen Planus and related diseases - 2k -> 2055
8. Seborrheic Keratoses and other Benign Tumors - 1.8k -> 1847
9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k -> 1702
1. Eczema 1677                                -> 1676
3. Atopic Dermatitis - 1.25k                  -> 1257


In [4]:
bad_files = []

# Scan every image file in every class folder
for cls in class_dirs:
    for img_path in cls.rglob("*"):
        if img_path.suffix.lower() not in image_exts:
            continue
        try:
            # verify() checks if file is broken/corrupt
            Image.open(img_path).verify()
        except Exception as e:
            bad_files.append((str(img_path.name), str(cls.name), str(e)))

print("Corrupted / unreadable images:", len(bad_files))

# Show first 10 bad files (safe: prints only file name + class, not full path)
for item in bad_files[:10]:
    print(item)


Corrupted / unreadable images: 0


In [5]:
# IMPORTANT: Only run this if you're sure you want to rename.
# You can also do this manually in Explorer.

rename_map = {
    "1. Eczema 1677": "eczema",
    "2. Melanoma 15.75k": "melanoma",
    "3. Atopic Dermatitis - 1.25k": "atopic_dermatitis",
    "4. Basal Cell Carcinoma (BCC) 3323": "basal_cell_carcinoma",
    "5. Melanocytic Nevi (NV) - 7970": "melanocytic_nevi",
    "6. Benign Keratosis-like Lesions (BKL) 2624": "benign_keratosis",
    "7. Psoriasis pictures Lichen Planus and related diseases - 2k": "psoriasis_lichen_planus",
    "8. Seborrheic Keratoses and other Benign Tumors - 1.8k": "seborrheic_keratosis",
    "9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k": "fungal_infections"
}

# Preview what would happen:
for old, new in rename_map.items():
    old_path = RAW_DIR / old
    if old_path.exists():
        print(f"Will rename: '{old}' -> '{new}'")
    else:
        print(f"Not found (skip): '{old}'")

# Uncomment to actually rename
# for old, new in rename_map.items():
#     old_path = RAW_DIR / old
#     new_path = RAW_DIR / new
#     if old_path.exists():
#         old_path.rename(new_path)


Will rename: '1. Eczema 1677' -> 'eczema'
Will rename: '2. Melanoma 15.75k' -> 'melanoma'
Will rename: '3. Atopic Dermatitis - 1.25k' -> 'atopic_dermatitis'
Will rename: '4. Basal Cell Carcinoma (BCC) 3323' -> 'basal_cell_carcinoma'
Will rename: '5. Melanocytic Nevi (NV) - 7970' -> 'melanocytic_nevi'
Will rename: '6. Benign Keratosis-like Lesions (BKL) 2624' -> 'benign_keratosis'
Will rename: '7. Psoriasis pictures Lichen Planus and related diseases - 2k' -> 'psoriasis_lichen_planus'
Will rename: '8. Seborrheic Keratoses and other Benign Tumors - 1.8k' -> 'seborrheic_keratosis'
Will rename: '9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k' -> 'fungal_infections'


In [6]:
import shutil

# Create output folders
splits = {"train": 0.7, "val": 0.15, "test": 0.15}

# Fresh start (optional): remove old split folder if it exists
# Uncomment if you want to regenerate splits from scratch.
# if OUTPUT_DIR.exists():
#     shutil.rmtree(OUTPUT_DIR)

for split in splits:
    (OUTPUT_DIR / split).mkdir(parents=True, exist_ok=True)

# Re-read class dirs (in case you renamed)
class_dirs = [p for p in RAW_DIR.iterdir() if p.is_dir()]

# Copy images into train/val/test per class
for cls in class_dirs:
    # Collect image files for this class
    imgs = [p for p in cls.rglob("*") if p.suffix.lower() in image_exts]
    random.shuffle(imgs)

    n = len(imgs)
    n_train = int(splits["train"] * n)
    n_val = int(splits["val"] * n)

    split_lists = {
        "train": imgs[:n_train],
        "val": imgs[n_train:n_train + n_val],
        "test": imgs[n_train + n_val:]
    }

    # Create class folders in each split
    for split in split_lists:
        (OUTPUT_DIR / split / cls.name).mkdir(parents=True, exist_ok=True)

    # Copy files
    for split, files in split_lists.items():
        for f in files:
            dst = OUTPUT_DIR / split / cls.name / f.name
            shutil.copy2(f, dst)

    print(f"Done split for class '{cls.name}' (total {n})")


Done split for class '1. Eczema 1677' (total 1676)
Done split for class '10. Warts Molluscum and other Viral Infections - 2103' (total 2103)
Done split for class '2. Melanoma 15.75k' (total 3140)
Done split for class '3. Atopic Dermatitis - 1.25k' (total 1257)
Done split for class '4. Basal Cell Carcinoma (BCC) 3323' (total 3323)
Done split for class '5. Melanocytic Nevi (NV) - 7970' (total 7970)
Done split for class '6. Benign Keratosis-like Lesions (BKL) 2624' (total 2079)
Done split for class '7. Psoriasis pictures Lichen Planus and related diseases - 2k' (total 2055)
Done split for class '8. Seborrheic Keratoses and other Benign Tumors - 1.8k' (total 1847)
Done split for class '9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k' (total 1702)


In [7]:
# Verify counts in new split folders
for split in ["train", "val", "test"]:
    split_dir = OUTPUT_DIR / split
    total = len([p for p in split_dir.rglob("*") if p.suffix.lower() in image_exts])
    print(split, "total images:", total)

train total images: 19003
val total images: 4069
test total images: 4080


In [11]:
pip install torch torchvision torchaudio


   ---------------------------------------- 0.0/113.8 MB ? eta -:--:--
   ---------------------------------------- 1.0/113.8 MB 8.9 MB/s eta 0:00:13
    --------------------------------------- 1.8/113.8 MB 5.9 MB/s eta 0:00:19
   - -------------------------------------- 2.9/113.8 MB 5.3 MB/s eta 0:00:21
   - -------------------------------------- 3.7/113.8 MB 4.7 MB/s eta 0:00:24
   - -------------------------------------- 4.5/113.8 MB 4.5 MB/s eta 0:00:25
   - -------------------------------------- 5.5/113.8 MB 4.6 MB/s eta 0:00:24
   -- ------------------------------------- 6.3/113.8 MB 4.6 MB/s eta 0:00:24
   -- ------------------------------------- 7.6/113.8 MB 4.6 MB/s eta 0:00:24
   -- ------------------------------------- 8.4/113.8 MB 4.6 MB/s eta 0:00:24
   --- ------------------------------------ 9.4/113.8 MB 4.6 MB/s eta 0:00:23
   --- ------------------------------------ 10.5/113.8 MB 4.6 MB/s eta 0:00:23
   ---- ----------------------------------- 11.5/113.8 MB 4.6 MB/s eta

In [12]:
import sys
print(sys.executable)
print(sys.version)


c:\Users\ACER\anaconda3\python.exe
3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]


In [13]:
import torch
print("torch version:", torch.__version__)
print("cuda available?", torch.cuda.is_available())

torch version: 2.10.0+cpu
cuda available? False


In [14]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Basic transforms (small test)
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_ds = datasets.ImageFolder(OUTPUT_DIR / "train", transform=train_tfms)
val_ds = datasets.ImageFolder(OUTPUT_DIR / "val", transform=val_tfms)

print("Classes:", train_ds.classes)
print("Train size:", len(train_ds))
print("Val size:", len(val_ds))

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0)


Classes: ['1. Eczema 1677', '10. Warts Molluscum and other Viral Infections - 2103', '2. Melanoma 15.75k', '3. Atopic Dermatitis - 1.25k', '4. Basal Cell Carcinoma (BCC) 3323', '5. Melanocytic Nevi (NV) - 7970', '6. Benign Keratosis-like Lesions (BKL) 2624', '7. Psoriasis pictures Lichen Planus and related diseases - 2k', '8. Seborrheic Keratoses and other Benign Tumors - 1.8k', '9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k']
Train size: 19003
Val size: 4069


In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

device = torch.device("cpu")

# ✅ Offline-safe: no download
model = models.efficientnet_b0(weights=None)

# Replace final classifier for your number of classes
num_classes = len(train_ds.classes)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

def run_one_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        if train:
            optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        if train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

# ✅ Smoke test: run just 1 epoch
train_loss, train_acc = run_one_epoch(train_loader, train=True)
val_loss, val_acc = run_one_epoch(val_loader, train=False)

print(f"Train: loss={train_loss:.4f}, acc={train_acc:.4f}")
print(f"Val:   loss={val_loss:.4f}, acc={val_acc:.4f}")


Train: loss=1.5364, acc=0.4444
Val:   loss=1.7888, acc=0.5424
